In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from urllib.request import urlretrieve

# hyperparameters
batch_size = 32
block_size = 8
max_iters = 5000
eval_interval = 300
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32
# ------------------------

torch.manual_seed(1337)

# read in data
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
try:
    data_path = Path(__file__).resolve().parent / "input.txt"
except NameError:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "input.txt",
        cwd / "gptBilds" / "gpt" / "input.txt",
        cwd.parent / "gptBilds" / "gpt" / "input.txt",
    ]
    data_path = next((path for path in candidates if path.exists()), candidates[1])

data_path.parent.mkdir(parents=True, exist_ok=True)
if not data_path.exists():
    urlretrieve(url, data_path)

with open(data_path, 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {s: i for i, s in enumerate(chars)}
itos = {i: s for i, s in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


xb, yb = get_batch('train')
print('xb shape:', xb.shape)
print('yb shape:', yb.shape)
print('device:', device)

xb shape: torch.Size([32, 8])
yb shape: torch.Size([32, 8])
device: cpu


In [6]:
class LayerNorm(nn.Module):

    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        xmean = x.mean(-1, keepdim=True)
        xvar = x.var(-1, keepdim=True, unbiased=False)
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        return self.gamma * xhat + self.beta


class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd)
        )

    def forward(self, x):
        return self.net(x)


# A simple tensor to play with before wiring up the whole model.
x = torch.randn(batch_size, block_size, n_embd, device=device)
print('x shape:', x.shape)
print('FeedForward(x) shape:', FeedForward(n_embd).to(device)(x).shape)

x shape: torch.Size([32, 8, 32])
FeedForward(x) shape: torch.Size([32, 8, 32])


In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.num_heads = num_heads
        self.head_size = head_size

        self.key = nn.Linear(n_embd, num_heads * head_size, bias=False)
        self.query = nn.Linear(n_embd, num_heads * head_size, bias=False)
        self.value = nn.Linear(n_embd, num_heads * head_size, bias=False)

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.proj = nn.Linear(num_heads * head_size, n_embd)

    def forward(self, x):
        B, T, C = x.shape  # x: (B, T, n_embd)

        # After the linear layer: (B, T, num_heads * head_size)
        # After view: (B, T, num_heads, head_size)
        # After transpose: (B, num_heads, T, head_size)
        k = self.key(x).view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        q = self.query(x).view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        v = self.value(x).view(B, T, self.num_heads, self.head_size).transpose(1, 2)

        # q: (B, num_heads, T, head_size)
        # k.transpose(-2, -1): (B, num_heads, head_size, T)
        # wei: (B, num_heads, T, T)
        wei = q @ k.transpose(-2, -1) * (self.head_size ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # mask broadcasts to (B, num_heads, T, T)
        wei = F.softmax(wei, dim=-1)  # still (B, num_heads, T, T)

        # wei @ v: (B, num_heads, T, T) @ (B, num_heads, T, head_size)
        # gives out: (B, num_heads, T, head_size)
        out = wei @ v

        # transpose back to (B, T, num_heads, head_size)
        # then flatten heads to (B, T, num_heads * head_size)
        out = out.transpose(1, 2).contiguous().view(B, T, self.num_heads * self.head_size)
        out = self.proj(out)  # (B, T, n_embd)
        return out


mha = MultiHeadAttention(num_heads=4, head_size=n_embd // 4).to(device)
out = mha(x)
print('mha output shape:', out.shape)
print('expected shape:', (batch_size, block_size, n_embd))

In [ ]:
# Optional once the model cell works:
# model = BigramLanguageModel().to(device)
# xb, yb = get_batch('train')
# logits, loss = model(xb, yb)
# print('logits shape:', logits.shape)
# print('loss:', loss.item())

# Training loop stays commented so the notebook remains easy to experiment with.
# for iter in range(max_iters):
#     if iter % eval_interval == 0:
#         losses = estimate_loss()
#         print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
#
#     xb, yb = get_batch('train')
#     logits, loss = model(xb, yb)
#     opt.zero_grad(set_to_none=True)
#     loss.backward()
#     opt.step()

In [ ]:
class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = LayerNorm(n_embd)
        self.ln2 = LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(x)
        x = x + self.ffwd(x)
        return x


class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4),
            LayerNorm(n_embd),
        )
        self.lm_head = nn.Linear(n_embd, vocab_size)
    
    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
